# Classical Image Segmentation

> **Intermediate · Classical vision**


## Why this matters

Segmentation assigns pixels to meaningful regions. It extends simple masks with strategies for touching objects and foreground extraction.

**Where it appears:** Cell/coin counting, foreground cutouts, product inspection, and interactive image editing.


## Learning Objectives

- Segment images using color thresholding, watershed, and GrabCut
- Understand the marker-based watershed algorithm for touching-object separation
- Know which segmentation method fits which problem shape


## Prerequisites

09 Thresholding and Morphology; 10 Edges, Contours, and Shape Measurement

Work through the examples in order. Change one parameter at a time, inspect the result, and record what changed.


## Core OpenCV APIs

color masks, distance transform, watershed, `cv2.grabCut`

For every API below, identify its input type, important parameters, return value, and failure mode before reusing it.


## Conceptual Foundation


### Image Segmentation

Segmentation partitions an image into meaningful regions. Simple color/
threshold-based segmentation fails when objects touch or overlap. The
**watershed** algorithm treats intensity as topography and 'floods' from
markers to separate touching objects -- essential for splitting adjacent
blobs that simple contour-finding would merge into one. **GrabCut** is an
interactive foreground/background segmentation method using a rough
user-provided rectangle, refined automatically via graph cuts.


## Setup

Run this cell once. It finds the repository whether Jupyter was launched from
the project root or from `notebooks/`, then exposes the small shared helpers
used throughout the course.


In [ ]:
import os
import sys
from pathlib import Path

os.environ.setdefault("MPLBACKEND", "Agg")

_candidates = (Path.cwd(), Path.cwd().parent)
REPO_ROOT = next(
    (path for path in _candidates if (path / "utils" / "cv_utils.py").exists()),
    None,
)
if REPO_ROOT is None:
    raise RuntimeError("Run this notebook from the repository root or notebooks/ directory.")

UTILS_DIR = REPO_ROOT / "utils"
if str(UTILS_DIR) not in sys.path:
    sys.path.insert(0, str(UTILS_DIR))

import cv2
import matplotlib.pyplot as plt
import numpy as np

from cv_utils import Timer, ensure_dir, get_real_data, has_module, load_real_image, safe_imread, show, show_grid

print("OpenCV:", cv2.__version__)
print("Repository:", REPO_ROOT)


## Guided Lessons


### 1. The touching-objects problem

Simple thresholding + contours merges two touching circular blobs into a single contour with no way to recover object count -- this motivates watershed.


In [ ]:
import cv2
import numpy as np
from cv_utils import load_real_image, get_real_data, show_grid


def touching_blobs() -> np.ndarray:
    img = load_real_image("images/objects", "coins.jpg")
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    # Otsu thresholding (coins are brighter than bg? Actually background is bright. Let's use simple threshold)
    _, binary = cv2.threshold(gray, 180, 255, cv2.THRESH_BINARY_INV)
    # Fill holes
    kernel = np.ones((7, 7), np.uint8)
    solid = cv2.morphologyEx(binary, cv2.MORPH_CLOSE, kernel, iterations=4)
    # Erode slightly so they barely touch
    solid = cv2.erode(solid, np.ones((3, 3), np.uint8), iterations=2)
    return solid


blobs = touching_blobs()
contours, _ = cv2.findContours(blobs, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
print(f"Naive contour count: {len(contours)} (Many coins merged!)")
show_grid([("touching blobs (coins)", blobs)])

### 2. Watershed segmentation to separate them

Use the distance transform to find blob centers as markers, then let watershed flood from those markers to correctly split the touching region into two labels.


In [ ]:
def watershed_split(binary_mask: np.ndarray) -> np.ndarray:
    """Split touching blobs in `binary_mask` using distance-transform markers + watershed.
    Returns a label image where each separated blob has a distinct positive integer id."""
    dist = cv2.distanceTransform(binary_mask, cv2.DIST_L2, 5)
    _, sure_fg = cv2.threshold(dist, 0.5 * dist.max(), 255, 0)
    sure_fg = sure_fg.astype(np.uint8)

    n_markers, markers = cv2.connectedComponents(sure_fg)
    markers = markers + 1  # background becomes 1, not 0
    unknown = cv2.subtract(binary_mask, sure_fg)
    markers[unknown == 255] = 0  # unknown region, to be resolved by watershed

    color_for_watershed = cv2.cvtColor(binary_mask, cv2.COLOR_GRAY2BGR)
    cv2.watershed(color_for_watershed, markers)
    return markers


labels = watershed_split(blobs)
object_ids = [
    v for v in np.unique(labels) if v > 1
]  # exclude background(1) and boundary(-1)
print(f"Watershed recovered object count: {len(object_ids)}  (correct: 2)")

visualization = cv2.cvtColor(blobs, cv2.COLOR_GRAY2BGR)
visualization[labels == -1] = (0, 0, 255)  # draw the watershed boundary line
show_grid([("watershed boundary drawn between the two blobs", visualization)])

### 3. GrabCut for foreground extraction

GrabCut only needs a rough bounding rectangle around the foreground object and refines the exact boundary automatically -- useful for background removal.


In [ ]:
from cv_utils import load_real_image, get_real_data


def grabcut_foreground(
    image: np.ndarray, rect: tuple[int, int, int, int]
) -> np.ndarray:
    """rect = (x, y, w, h), a rough box around the foreground object."""
    mask = np.zeros(image.shape[:2], np.uint8)
    bgd_model = np.zeros((1, 65), np.float64)
    fgd_model = np.zeros((1, 65), np.float64)

    cv2.grabCut(
        image, mask, rect, bgd_model, fgd_model, iterCount=5, mode=cv2.GC_INIT_WITH_RECT
    )
    binary_mask = np.where(
        (mask == cv2.GC_FGD) | (mask == cv2.GC_PR_FGD), 255, 0
    ).astype(np.uint8)
    return binary_mask


scene = load_real_image("images/objects", "smarties.png")
rough_rect = (260, 30, 140, 140)  # loosely around the green circle
fg_mask = grabcut_foreground(scene, rough_rect)
extracted = cv2.bitwise_and(scene, scene, mask=fg_mask)

show_grid(
    [
        ("original + rough rect", scene),
        ("GrabCut mask", fg_mask),
        ("extracted foreground", extracted),
    ]
)

## Mini Projects

Complete one project unaided before reading any provided solution. Extend it with a parameter, dataset, or failure case of your own.


### Mini Project — Image Segmentation: Marker-Controlled Watershed Segmentation

Standard watershed segmentation often over-segments objects containing uneven internal gradients. To prevent this, we use marker-controlled watershed segmentation, where manual or calculated markers define foreground seed regions.


In [ ]:
# Load touching coins mask
img = touching_blobs()

# Compute distance transform
dist_transform = cv2.distanceTransform(img, cv2.DIST_L2, 5)

# Binarize distance transform to isolate local maximums (markers)
_, markers_fg = cv2.threshold(dist_transform, 0.5 * dist_transform.max(), 255, 0)
markers_fg = markers_fg.astype(np.uint8)

# Find connected components for markers
_, markers = cv2.connectedComponents(markers_fg)
markers = markers + 1
unknown = cv2.subtract(img, markers_fg)
markers[unknown == 255] = 0

# Apply Watershed
color_src = cv2.cvtColor(img, cv2.COLOR_GRAY2BGR)
cv2.watershed(color_src, markers)

print("Marker-controlled watershed completed.")
show_grid(
    [
        ("Touching Objects", img),
        ("Foreground Seeds (Markers)", markers_fg),
        ("Segmented Boundaries", (markers == -1).astype(np.uint8) * 255),
    ]
)

## Exercises

Attempt the beginner, intermediate, and advanced prompts in order. Keep notes on assumptions and failures, not just successful output.


### Exercises — Image Segmentation
1. Add a third overlapping circle to `touching_blobs` and confirm `watershed_split` still recovers the correct count.
2. Compare GrabCut's result quality with a rect that's too tight vs generously oversized around the object.
3. Combine HSV color thresholding (notebook 07) with watershed to segment same-colored touching objects.

Use the empty cell below to work through them.


#### Solutions — Image Segmentation

In [ ]:
# Solution 1: Plot the watershed boundaries directly on the original image
def segment_and_plot() -> None:
    img = load_real_image("images/objects", "coins.jpg")
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    _, binary = cv2.threshold(gray, 180, 255, cv2.THRESH_BINARY_INV)
    kernel = np.ones((7, 7), np.uint8)
    binary = cv2.morphologyEx(binary, cv2.MORPH_CLOSE, kernel, iterations=4)
    dist = cv2.distanceTransform(binary, cv2.DIST_L2, 5)
    _, fg = cv2.threshold(dist, 0.5 * dist.max(), 255, 0)
    _, markers = cv2.connectedComponents(fg.astype(np.uint8))
    markers = markers + 1
    markers[cv2.subtract(binary, fg.astype(np.uint8)) == 255] = 0
    cv2.watershed(img, markers)

    annotated = img.copy()
    annotated[markers == -1] = [0, 0, 255]
    show_grid([("Watershed Segments on Original", annotated)])
    print("Recovered marker seeds count:", markers.max() - 1)

In [ ]:
# Solution 2: GrabCut rect tightness comparison
# Explanation: GrabCut estimates foreground color models from pixels inside the user-defined rect.
# If the bounding box is too tight, it crops out parts of the object, leading to missing segments.
# If the bounding box is too loose, background pixels contaminate the foreground color models,
# resulting in background segments included in the output.


In [ ]:
# Solution 3: Combine HSV color thresholding with watershed
def segment_by_color_and_watershed(image: np.ndarray) -> np.ndarray:
    """Locate target seeds by color range, then apply watershed separation."""
    hsv = cv2.cvtColor(image, cv2.COLOR_BGR2HSV)
    # Filter green range
    mask = cv2.inRange(hsv, np.array([35, 50, 50]), np.array([85, 255, 255]))

    # Isolate markers
    dist = cv2.distanceTransform(mask, cv2.DIST_L2, 5)
    _, fg = cv2.threshold(dist, 0.5 * dist.max(), 255, 0)
    _, markers = cv2.connectedComponents(fg.astype(np.uint8))

    return markers


# Run validation test
segment_and_plot()

## Summary

You can choose between color-based, marker-based, and interactive segmentation and identify when the assumptions of each method fail.

- **Best Practices:** Create and inspect markers explicitly, use morphology before watershed, and keep user interaction separate from the segmentation algorithm.
- **Common Pitfalls:** Expecting a generic method to understand semantics, using poor foreground/background seeds, and treating a mask as ground truth without evaluation.